In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json
import subprocess
import unicodedata
import zipfile

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/dataset_senales_acusticas")

RAW_AUDIO_DIR = BASE_DIR / "raw_audio"
AUDIO_DIR = BASE_DIR / "audio_32khz"
SOURCE_DIR = BASE_DIR / "source_metadata"
METADATA_DIR = BASE_DIR / "metadata"
STATISTICS_DIR = BASE_DIR / "statistics"

PTERO_DIR = SOURCE_DIR / "PteroSet"
COLCR_DIR = SOURCE_DIR / "Col-CR"
PERU_DIR = SOURCE_DIR / "Peru"
WABAD_DIR = SOURCE_DIR / "WABAD-CO"

EVENTS_BASE_PATH = METADATA_DIR / "events_manifest_base.csv"
RECORDINGS_BASE_PATH = METADATA_DIR / "recordings_manifest_base.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_FOLDERS = {
    "PteroSet": AUDIO_DIR / "pteroset",
    "Col-CR": AUDIO_DIR / "colcr",
    "Peru": AUDIO_DIR / "peru",
    "WABAD-CO": AUDIO_DIR / "wabad_co",
}

RAW_FOLDERS = {
    "PteroSet": RAW_AUDIO_DIR / "pteroset",
    "Col-CR": RAW_AUDIO_DIR / "colcr",
    "Peru": RAW_AUDIO_DIR / "peru",
    "WABAD-CO": RAW_AUDIO_DIR / "wabad_co",
}

ZENODO_RECORDS = {
    "PteroSet": 21829388,
    "Col-CR": 7525349,
    "Peru": 7079124,
    "WABAD-CO": 20513304,
}

PTERO_PROJECTS = {"MAP1", "PPA1", "PPA2", "PPA3", "PPA4"}
WABAD_CO_SITES = {"BRE", "CARI", "SAL", "SPMCO", "TAM", "UNI", "VER"}
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a",".aac", ".wma", ".aif", ".aiff",}

TARGET_SAMPLE_RATE = 32000
TARGET_CHANNELS = 1

print("Proyecto:", BASE_DIR)

## Validación de entradas

In [ ]:
SOURCE_FILES = {
    "PteroSet annotations": PTERO_DIR / "annotations_identification.json",
    "PteroSet species annotations": PTERO_DIR / "annotations_species.json",
    "PteroSet species": PTERO_DIR / "species.csv",
    "Col-CR annotations": COLCR_DIR / "annotations.csv",
    "Col-CR species": COLCR_DIR / "species.csv",
    "Peru annotations": PERU_DIR / "annotations.csv",
    "Peru species": PERU_DIR / "species.csv",
    "WABAD annotations": WABAD_DIR / "Pooled annotations.csv",
    "WABAD metadata": WABAD_DIR / "Metadata.csv",
}

for name, path in SOURCE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró {name}: {path}. "
            "Ejecuta primero el notebook 00."
        )
    print(f"OK: {name}")

for dataset, folder in AUDIO_FOLDERS.items():
    if not folder.exists():
        raise FileNotFoundError(
            f"No existe la carpeta de audio de {dataset}: {folder}"
        )

print("\nEntradas mínimas disponibles.")

## Funciones auxiliares

`recording_uid` se construye a partir del dataset y del **stem** normalizado del nombre de archivo. Así `.wav`, `.WAV` y `.flac` no generan identidades distintas para la misma grabación.

In [ ]:
TAXON_FIXES = {
    "melanerpes cruentats": "melanerpes cruentatus",
    "megarynchus pitagua": "megarynchus pitangua",
    "mesembrinibis cayanensis": "mesembrinibis cayennensis",
}

def pick_col(df, candidates, required=True):
    for column in candidates:
        if column in df.columns:
            return column

    if required:
        raise KeyError(
            f"No se encontró ninguna de {candidates}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    return None

def normalize_species(series):
    return (
        series.astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
        .replace(TAXON_FIXES)
        .replace({
            "????": pd.NA,
            "unknown": pd.NA,
            "unidentified": pd.NA,
            "unidentified species": pd.NA,
            "": pd.NA,
        })
    )

def recording_key(value):
    if pd.isna(value):
        return pd.NA

    stem = Path(str(value)).stem
    return (
        unicodedata.normalize("NFKC", stem)
        .strip()
        .casefold()
    )

def add_common_event_columns(df):
    df = df.copy()
    df["species_raw"] = df["species_raw"].astype("string")
    df["species_source_code"] = (df["species_source_code"].astype("string"))
    df["species_scientific"] = normalize_species(df["species_scientific"])
    df["identified"] = df["species_scientific"].notna()
    for column in ["t_min", "t_max", "f_min_original", "f_max_original",]:
        df[column] = pd.to_numeric(df[column], errors="coerce",)

    # El corpus de modelado está normalizado a 32 kHz.
    # Nyquist = 16 kHz.
    df["f_min"] = df["f_min_original"].clip(lower=0,upper=16000,)

    df["f_max"] = df["f_max_original"].clip(lower=0,upper=16000,)

    df["duration_event_sec"] = (df["t_max"] - df["t_min"])

    df["bandwidth_hz"] = (df["f_max"] - df["f_min"])

    df["use_detector"] = (
        df["t_min"].notna()
        & df["t_max"].notna()
        & df["f_min"].notna()
        & df["f_max"].notna()
        & (df["t_min"] >= 0)
        & (df["f_min"] >= 0)
        & (df["t_max"] > df["t_min"])
        & (df["f_max"] > df["f_min"])
    )

    df["recording_key"] = (
        df["recording"].map(recording_key)
    )

    if df["recording_key"].isna().any():
        raise ValueError(
            "Hay eventos sin recording_key."
        )

    df["recording_uid"] = (
        df["dataset"].astype(str)
        + "::"
        + df["recording_key"].astype(str)
    )

    return df


def ffprobe_audio(path):
    command = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=sample_rate,channels",
        "-show_entries", "format=duration",
        "-of", "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        return None

    try:
        data = json.loads(result.stdout)
        stream = data["streams"][0]

        return {
            "sample_rate": int(stream["sample_rate"]),
            "channels": int(stream["channels"]),
            "duration_sec": float(
                data["format"].get("duration", "nan")
            ),
        }

    except Exception:
        return None

# 1. Inventario físico de audios

Se inspeccionan los ZIP fuente sin extraerlos permanentemente. Esto permite conocer qué grabaciones existen físicamente y a qué archivo fuente pertenecen.

Esperamos:

- PteroSet: 563 audios
- Col-CR: 34
- Perú: 21
- WABAD-CO: 399

Total físico: **1017**.

In [ ]:
ARCHIVES = []

for project in sorted(PTERO_PROJECTS):
    ARCHIVES.append({
        "dataset": "PteroSet",
        "source_group": project,
        "source_archive": f"{project}.zip",
        "country_default": "Colombia",
    })

ARCHIVES.append({
    "dataset": "Col-CR",
    "source_group": None,
    "source_archive": "soundscape_data.zip",
    "country_default": None,
})

ARCHIVES.append({
    "dataset": "Peru",
    "source_group": None,
    "source_archive": "soundscape_data.zip",
    "country_default": "Peru",
})

for site in sorted(WABAD_CO_SITES):
    ARCHIVES.append({
        "dataset": "WABAD-CO",
        "source_group": site,
        "source_archive": f"{site}.zip",
        "country_default": "Colombia",
    })


inventory_rows = []

for spec in ARCHIVES:
    dataset = spec["dataset"]
    archive_path = (
        RAW_FOLDERS[dataset]
        / spec["source_archive"]
    )

    if not archive_path.exists():
        raise FileNotFoundError(
            f"No existe el ZIP esperado: {archive_path}"
        )

    with zipfile.ZipFile(archive_path, "r") as zf:
        members = [
            info.filename
            for info in zf.infolist()
            if (
                not info.is_dir()
                and Path(info.filename).suffix.lower()
                in AUDIO_EXTENSIONS
            )
        ]

    for member in members:
        source_recording = Path(member).name
        key = recording_key(source_recording)

        source_group = spec["source_group"]
        country = spec["country_default"]

        if dataset == "Col-CR":
            match = pd.Series([source_recording]).str.extract(
                r"_(S\d+)_",
                expand=False,
            ).iloc[0]

            source_group = match
            country = {
                "S01": "Colombia",
                "S02": "Costa Rica",
            }.get(match)

        elif dataset == "Peru":
            source_group = (
                pd.Series([source_recording])
                .str.extract(
                    r"_(S\d+)_",
                    expand=False,
                )
                .iloc[0]
            )

        inventory_rows.append({
            "dataset": dataset,
            "recording_key": key,
            "recording_uid": f"{dataset}::{key}",
            "source_recording": source_recording,
            "source_member": member,
            "source_group": source_group,
            "source_archive": spec["source_archive"],
            "country": country,
            "record_id": ZENODO_RECORDS[dataset],
        })

audio_source_inventory = pd.DataFrame(
    inventory_rows
)

duplicate_keys = (
    audio_source_inventory
    .groupby(["dataset", "recording_key"])
    .size()
)

duplicate_keys = duplicate_keys[
    duplicate_keys > 1
]

if len(duplicate_keys):
    display(duplicate_keys)
    raise ValueError(
        "Hay stems duplicados dentro del mismo dataset."
    )

audio_source_inventory.to_csv(
    STATISTICS_DIR / "audio_source_inventory.csv",
    index=False,
)

display(
    audio_source_inventory
    .groupby("dataset")
    .agg(
        audios_fuente=("recording_uid", "nunique"),
        archives=("source_archive", "nunique"),
    )
)

## Reconciliación con `audio_32khz/`

In [ ]:
normalized_rows = []

for dataset, folder in AUDIO_FOLDERS.items():
    for path in sorted(folder.glob("*.flac")):
        key = recording_key(path.name)

        normalized_rows.append({
            "dataset": dataset,
            "recording_key": key,
            "recording_uid": f"{dataset}::{key}",
            "normalized_file": path.name,
            "normalized_path": (
                Path("audio_32khz")
                / folder.name
                / path.name
            ).as_posix(),
        })

normalized_inventory = pd.DataFrame(
    normalized_rows
)

if normalized_inventory.duplicated(
    ["dataset", "recording_key"]
).any():
    raise ValueError(
        "Hay FLAC normalizados con recording_key duplicado."
    )

source_keys = set(
    zip(
        audio_source_inventory["dataset"],
        audio_source_inventory["recording_key"],
    )
)

normalized_keys = set(
    zip(
        normalized_inventory["dataset"],
        normalized_inventory["recording_key"],
    )
)

missing_normalized = source_keys - normalized_keys
unexpected_normalized = normalized_keys - source_keys

print("Audios fuente sin FLAC:", len(missing_normalized))
print("FLAC sin audio fuente:", len(unexpected_normalized))

assert not missing_normalized
assert not unexpected_normalized

physical_audio = (
    audio_source_inventory.merge(
        normalized_inventory,
        on=[
            "dataset",
            "recording_key",
            "recording_uid",
        ],
        how="inner",
        validate="one_to_one",
    )
)

print("Audios físicos reconciliados:", len(physical_audio))
display(
    physical_audio.groupby("dataset").size().rename("audios")
)

## Duración y propiedades de los FLAC

Se reutiliza `normalized_audio_validation.csv` generado por el notebook 00. Si no existe, se reconstruye con `ffprobe`.

In [ ]:
validation_path = (
    STATISTICS_DIR
    / "normalized_audio_validation.csv"
)

if validation_path.exists():
    normalized_validation = pd.read_csv(
        validation_path
    )

    normalized_validation["recording_key"] = (
        normalized_validation["file"].map(
            recording_key
        )
    )

else:
    rows = []

    for _, row in normalized_inventory.iterrows():
        path = BASE_DIR / row["normalized_path"]
        info = ffprobe_audio(path)

        rows.append({
            "dataset": row["dataset"],
            "file": row["normalized_file"],
            "recording_key": row["recording_key"],
            "valid": (
                info is not None
                and info["sample_rate"] == 32000
                and info["channels"] == 1
                and info["duration_sec"] > 0
            ),
            "sample_rate": (
                info["sample_rate"]
                if info else np.nan
            ),
            "channels": (
                info["channels"]
                if info else np.nan
            ),
            "duration_sec": (
                info["duration_sec"]
                if info else np.nan
            ),
        })

    normalized_validation = pd.DataFrame(rows)

    normalized_validation.to_csv(
        validation_path,
        index=False,
    )

if normalized_validation["valid"].dtype != bool:
    normalized_validation["valid"] = (
        normalized_validation["valid"]
        .astype(str)
        .str.casefold()
        .map({
            "true": True,
            "false": False,
        })
    )

assert normalized_validation["valid"].notna().all()
assert normalized_validation["valid"].all()

technical = (
    normalized_validation[
        [
            "dataset",
            "recording_key",
            "duration_sec",
            "sample_rate",
            "channels",
        ]
    ]
    .rename(columns={
        "sample_rate": "normalized_sample_rate",
        "channels": "normalized_channels",
    })
)

physical_audio = physical_audio.merge(
    technical,
    on=["dataset", "recording_key"],
    how="left",
    validate="one_to_one",
)

assert physical_audio["duration_sec"].notna().all()
assert (
    physical_audio["normalized_sample_rate"]
    == 32000
).all()
assert (
    physical_audio["normalized_channels"]
    == 1
).all()

display(
    physical_audio.groupby("dataset").agg(
        audios=("recording_uid", "size"),
        horas=(
            "duration_sec",
            lambda x: x.sum() / 3600,
        ),
    )
)

# 2. PteroSet

In [ ]:
with open(
    PTERO_DIR / "annotations_identification.json",
    encoding="utf-8",
) as f:
    ptero_ident_json = json.load(f)

with open(
    PTERO_DIR / "annotations_species.json",
    encoding="utf-8",
) as f:
    ptero_species_json = json.load(f)

ptero_species_table = pd.read_csv(
    PTERO_DIR / "species.csv"
)

ident_annotations = pd.DataFrame(
    ptero_ident_json["annotations"]
)

species_annotations = pd.DataFrame(
    ptero_species_json["annotations"]
)

sounds = pd.DataFrame(
    ptero_ident_json["sounds"]
)

categories = pd.DataFrame(
    ptero_species_json["categories"]
)

sound_info = (
    sounds.set_index("id").to_dict("index")
)

category_code = (
    categories
    .set_index("id")["name"]
    .to_dict()
)

species_code_col = pick_col(
    ptero_species_table,
    [
        "code",
        "species_code",
        "eBird Code",
        "ebird_code",
    ],
)

species_name_col = pick_col(
    ptero_species_table,
    [
        "species",
        "scientific_name",
        "Scientific Name",
    ],
)

code_to_species = dict(
    zip(
        ptero_species_table[species_code_col],
        ptero_species_table[species_name_col],
    )
)

species_by_annotation_id = {}

has_annotation_id = (
    "id" in species_annotations.columns
    and "id" in ident_annotations.columns
)

if has_annotation_id:
    for _, annotation in species_annotations.iterrows():
        source_code = category_code.get(
            annotation["category_id"]
        )

        source_name = code_to_species.get(
            source_code,
            source_code,
        )

        species_by_annotation_id[
            annotation["id"]
        ] = (source_code, source_name)

    shared = (
        ident_annotations["id"]
        .isin(species_by_annotation_id)
        .sum()
    )

    use_geometry = (
        shared != len(species_annotations)
    )

else:
    use_geometry = True

print("Emparejamiento por geometría:", use_geometry)

species_by_geometry = {}

if use_geometry:
    for _, annotation in species_annotations.iterrows():
        source_code = category_code.get(
            annotation["category_id"]
        )

        source_name = code_to_species.get(
            source_code,
            source_code,
        )

        key = (
            annotation["sound_id"],
            annotation["t_min"],
            annotation["t_max"],
            annotation["f_min"],
            annotation["f_max"],
        )

        value = (source_code, source_name)

        if (
            key in species_by_geometry
            and species_by_geometry[key] != value
        ):
            raise ValueError(
                "Conflicto taxonómico en una misma geometría PteroSet."
            )

        species_by_geometry[key] = value

sound_path_col = pick_col(
    sounds,
    [
        "file_name_path",
        "file_name",
        "filename",
        "path",
    ],
)

rows = []

for i, annotation in ident_annotations.iterrows():
    sound = sound_info[
        annotation["sound_id"]
    ]

    sound_path = sound[sound_path_col]
    recording = Path(str(sound_path)).name

    project = sound.get("project")

    if pd.isna(project) or not project:
        matches = [
            p
            for p in sorted(PTERO_PROJECTS)
            if p in str(sound_path)
        ]

        if len(matches) != 1:
            raise ValueError(
                f"No se pudo inferir proyecto para {sound_path}"
            )

        project = matches[0]

    if use_geometry:
        key = (
            annotation["sound_id"],
            annotation["t_min"],
            annotation["t_max"],
            annotation["f_min"],
            annotation["f_max"],
        )

        source_code, source_name = (
            species_by_geometry.get(
                key,
                (pd.NA, pd.NA),
            )
        )

    else:
        source_code, source_name = (
            species_by_annotation_id.get(
                annotation["id"],
                (pd.NA, pd.NA),
            )
        )

    rows.append({
        "event_id": f"PteroSet::{i}",
        "dataset": "PteroSet",
        "recording": recording,
        "source_group": str(project),
        "source_archive": f"{project}.zip",
        "country": "Colombia",
        "species_source_code": source_code,
        "species_raw": source_name,
        "species_scientific": source_name,
        "t_min": annotation["t_min"],
        "t_max": annotation["t_max"],
        "f_min_original": annotation["f_min"],
        "f_max_original": annotation["f_max"],
    })

pteroset = add_common_event_columns(
    pd.DataFrame(rows)
)

assert len(pteroset) == 15372
assert pteroset["recording_uid"].nunique() == 563
assert int(pteroset["identified"].sum()) == 6702

print("Eventos:", len(pteroset))
print("Grabaciones:", pteroset["recording_uid"].nunique())
print("Identificados:", int(pteroset["identified"].sum()))

# 3. Col-CR

Se conserva el dataset completo. `S01` corresponde a Colombia y `S02` a Costa Rica.

In [ ]:
colcr_annotations = pd.read_csv(
    COLCR_DIR / "annotations.csv"
)

colcr_species = pd.read_csv(
    COLCR_DIR / "species.csv"
)

file_col = pick_col(
    colcr_annotations,
    ["Filename", "filename"],
)

annotation_code_col = pick_col(
    colcr_annotations,
    [
        "Species eBird Code",
        "species_code",
        "Species Code",
    ],
)

lookup_code_col = pick_col(
    colcr_species,
    [
        "Species eBird Code",
        "species_code",
        "Code",
        "code",
    ],
)

lookup_name_col = pick_col(
    colcr_species,
    [
        "Scientific Name",
        "scientific_name",
        "Species",
        "species",
    ],
)

code_to_species_colcr = dict(
    zip(
        colcr_species[lookup_code_col],
        colcr_species[lookup_name_col],
    )
)

species_names = (
    colcr_annotations[annotation_code_col]
    .map(code_to_species_colcr)
)

colcr = pd.DataFrame({
    "event_id": (
        "Col-CR::"
        + colcr_annotations.index.astype(str)
    ),
    "dataset": "Col-CR",
    "recording": (
        colcr_annotations[file_col]
        .astype(str)
        .map(lambda x: Path(x).name)
    ),
    "species_source_code": (
        colcr_annotations[annotation_code_col]
        .astype("string")
    ),
    "species_raw": species_names,
    "species_scientific": species_names,
    "t_min": colcr_annotations[
        pick_col(
            colcr_annotations,
            ["Start Time (s)", "start_time"],
        )
    ],
    "t_max": colcr_annotations[
        pick_col(
            colcr_annotations,
            ["End Time (s)", "end_time"],
        )
    ],
    "f_min_original": colcr_annotations[
        pick_col(
            colcr_annotations,
            ["Low Freq (Hz)", "low_freq"],
        )
    ],
    "f_max_original": colcr_annotations[
        pick_col(
            colcr_annotations,
            ["High Freq (Hz)", "high_freq"],
        )
    ],
})

colcr["source_group"] = (
    colcr["recording"].str.extract(
        r"_(S\d+)_",
        expand=False,
    )
)

colcr["source_archive"] = "soundscape_data.zip"

colcr["country"] = (
    colcr["source_group"].map({
        "S01": "Colombia",
        "S02": "Costa Rica",
    })
)

if colcr["country"].isna().any():
    display(
        colcr.loc[
            colcr["country"].isna(),
            ["recording", "source_group"],
        ].drop_duplicates()
    )

    raise ValueError(
        "No se pudo asignar país a algunas grabaciones Col-CR."
    )

colcr = add_common_event_columns(colcr)

assert len(colcr) == 7338
assert colcr["recording_uid"].nunique() == 34
assert int(colcr["identified"].sum()) == 6952

print("Eventos:", len(colcr))
print("Grabaciones:", colcr["recording_uid"].nunique())
print("Identificados:", int(colcr["identified"].sum()))

# 4. Southwestern Amazon Basin — Perú

In [ ]:
peru_annotations = pd.read_csv(
    PERU_DIR / "annotations.csv"
)

peru_species = pd.read_csv(
    PERU_DIR / "species.csv"
)

file_col = pick_col(
    peru_annotations,
    ["Filename", "filename"],
)

annotation_code_col = pick_col(
    peru_annotations,
    [
        "Species eBird Code",
        "species_code",
        "Species Code",
    ],
)

lookup_code_col = pick_col(
    peru_species,
    [
        "Species eBird Code",
        "species_code",
        "Code",
        "code",
    ],
)

lookup_name_col = pick_col(
    peru_species,
    [
        "Scientific Name",
        "scientific_name",
        "Species",
        "species",
    ],
)

code_to_species_peru = dict(
    zip(
        peru_species[lookup_code_col],
        peru_species[lookup_name_col],
    )
)

species_names = (
    peru_annotations[annotation_code_col]
    .map(code_to_species_peru)
)

peru = pd.DataFrame({
    "event_id": (
        "Peru::"
        + peru_annotations.index.astype(str)
    ),
    "dataset": "Peru",
    "recording": (
        peru_annotations[file_col]
        .astype(str)
        .map(lambda x: Path(x).name)
    ),
    "country": "Peru",
    "species_source_code": (
        peru_annotations[annotation_code_col]
        .astype("string")
    ),
    "species_raw": species_names,
    "species_scientific": species_names,
    "t_min": peru_annotations[
        pick_col(
            peru_annotations,
            ["Start Time (s)", "start_time"],
        )
    ],
    "t_max": peru_annotations[
        pick_col(
            peru_annotations,
            ["End Time (s)", "end_time"],
        )
    ],
    "f_min_original": peru_annotations[
        pick_col(
            peru_annotations,
            ["Low Freq (Hz)", "low_freq"],
        )
    ],
    "f_max_original": peru_annotations[
        pick_col(
            peru_annotations,
            ["High Freq (Hz)", "high_freq"],
        )
    ],
})

peru["source_group"] = (
    peru["recording"].str.extract(
        r"_(S\d+)_",
        expand=False,
    )
)

peru["source_archive"] = "soundscape_data.zip"

peru = add_common_event_columns(peru)

assert len(peru) == 16482
assert peru["recording_uid"].nunique() == 21
assert int(peru["identified"].sum()) == 14798

print("Eventos:", len(peru))
print("Grabaciones:", peru["recording_uid"].nunique())
print("Identificados:", int(peru["identified"].sum()))

# 5. WABAD-CO

Solo se conservan anotaciones de:

`BRE, CARI, SAL, SPMCO, TAM, UNI, VER`.

Esperamos 9273 eventos en 393 grabaciones anotadas, aunque físicamente existan 399 audios.

In [ ]:
wabad_annotations = pd.read_csv(
    WABAD_DIR / "Pooled annotations.csv"
)

required = {
    "Species",
    "Site",
    "Recording",
    "Begin_Time_(s)",
    "End_Time_(s)",
    "Low_Freq_(Hz)",
    "High_Freq_(Hz)",
}

missing = required - set(
    wabad_annotations.columns
)

if missing:
    raise KeyError(
        f"Faltan columnas WABAD: {sorted(missing)}"
    )

wabad_source = (
    wabad_annotations.loc[
        wabad_annotations["Site"].isin(
            WABAD_CO_SITES
        )
    ]
    .copy()
)

wabad_co = pd.DataFrame({
    "event_id": (
        "WABAD-CO::"
        + wabad_source.index.astype(str)
    ),
    "dataset": "WABAD-CO",
    "recording": (
        wabad_source["Recording"]
        .astype(str)
        .map(lambda x: Path(x).name)
    ),
    "source_group": (
        wabad_source["Site"].astype(str)
    ),
    "source_archive": (
        wabad_source["Site"].astype(str)
        + ".zip"
    ),
    "country": "Colombia",
    "species_source_code": pd.NA,
    "species_raw": wabad_source["Species"],
    "species_scientific": wabad_source["Species"],
    "t_min": pd.to_numeric(
        wabad_source["Begin_Time_(s)"],
        errors="coerce",
    ),
    "t_max": pd.to_numeric(
        wabad_source["End_Time_(s)"],
        errors="coerce",
    ),
    "f_min_original": pd.to_numeric(
        wabad_source["Low_Freq_(Hz)"],
        errors="coerce",
    ),
    "f_max_original": pd.to_numeric(
        wabad_source["High_Freq_(Hz)"],
        errors="coerce",
    ),
})

wabad_co = add_common_event_columns(
    wabad_co
)

assert len(wabad_co) == 9273
assert wabad_co["recording_uid"].nunique() == 393
assert set(wabad_co["source_group"].unique()) == WABAD_CO_SITES

print("Eventos:", len(wabad_co))
print("Grabaciones anotadas:", wabad_co["recording_uid"].nunique())

# 6. Integración de `events_manifest_base`

In [ ]:
events_base = pd.concat(
    [
        pteroset,
        colcr,
        peru,
        wabad_co,
    ],
    ignore_index=True,
)

EVENT_COLUMNS = [
    "event_id",
    "dataset",
    "recording",
    "recording_key",
    "recording_uid",
    "source_group",
    "source_archive",
    "country",
    "species_source_code",
    "species_raw",
    "species_scientific",
    "identified",
    "t_min",
    "t_max",
    "f_min_original",
    "f_max_original",
    "f_min",
    "f_max",
    "duration_event_sec",
    "bandwidth_hz",
    "use_detector",
]

events_base = events_base[
    EVENT_COLUMNS
].copy()

physical_keys = set(
    zip(
        physical_audio["dataset"],
        physical_audio["recording_key"],
    )
)

event_keys = set(
    zip(
        events_base["dataset"],
        events_base["recording_key"],
    )
)

events_without_audio = (
    event_keys - physical_keys
)

if events_without_audio:
    print(sorted(events_without_audio)[:30])
    raise AssertionError(
        "Hay grabaciones anotadas sin audio físico."
    )

events_base["audio_available"] = True

display(
    events_base.groupby("dataset").agg(
        eventos=("event_id", "size"),
        grabaciones=("recording_uid", "nunique"),
        identificados=("identified", "sum"),
        detector_validos=("use_detector", "sum"),
    )
)

# 7. Reconciliación audio ↔ anotaciones

Los seis audios WABAD sin anotaciones se conservan en el inventario físico, pero **no se convierten en negativos** ni en background.

In [ ]:
annotation_counts = (
    events_base.groupby(
        ["dataset", "recording_key"],
        as_index=False,
    )
    .agg(
        annotation_event_count=("event_id", "size"),
        identified_event_count=("identified", "sum"),
        detector_valid_event_count=("use_detector", "sum"),
    )
)

reconciliation = physical_audio.merge(
    annotation_counts,
    on=["dataset", "recording_key"],
    how="left",
    validate="one_to_one",
)

for column in [
    "annotation_event_count",
    "identified_event_count",
    "detector_valid_event_count",
]:
    reconciliation[column] = (
        reconciliation[column]
        .fillna(0)
        .astype(int)
    )

reconciliation["has_annotations"] = (
    reconciliation["annotation_event_count"] > 0
)

reconciliation["status"] = np.where(
    reconciliation["has_annotations"],
    "annotated",
    "audio_only",
)

reconciliation.to_csv(
    STATISTICS_DIR
    / "audio_annotation_reconciliation.csv",
    index=False,
)

display(
    reconciliation.groupby(
        ["dataset", "status"]
    ).size().rename("grabaciones")
)

audio_only = reconciliation.loc[
    ~reconciliation["has_annotations"]
].copy()

print("\nAudios físicos sin anotaciones:", len(audio_only))

if len(audio_only):
    display(
        audio_only[
            [
                "dataset",
                "source_group",
                "source_archive",
                "source_recording",
                "normalized_path",
            ]
        ]
        .sort_values(
            ["dataset", "source_group", "source_recording"]
        )
        .reset_index(drop=True)
    )

## Validación específica de WABAD

In [ ]:
wabad_physical = reconciliation.loc[
    reconciliation["dataset"].eq("WABAD-CO")
]

wabad_annotated = wabad_physical.loc[
    wabad_physical["has_annotations"]
]

wabad_audio_only = wabad_physical.loc[
    ~wabad_physical["has_annotations"]
]

print("WABAD-CO físicos:", len(wabad_physical))
print("WABAD-CO anotados:", len(wabad_annotated))
print("WABAD-CO solo audio:", len(wabad_audio_only))

assert len(wabad_physical) == 399
assert len(wabad_annotated) == 393
assert len(wabad_audio_only) == 6

assert len(audio_only) == 6
assert set(audio_only["dataset"].unique()) == {"WABAD-CO"}

print("Reconciliación WABAD correcta.")

# 8. Construcción de `recordings_manifest_base.csv`

Este manifest incluye **todos los 1017 audios físicos**. Las rutas se guardan relativas a `dataset_senales_acusticas/`.

In [ ]:
recordings_base = reconciliation.copy()

recordings_base["audio_available"] = True

RECORDING_COLUMNS = [
    "recording_uid",
    "dataset",
    "recording_key",
    "source_recording",
    "source_group",
    "source_archive",
    "country",
    "record_id",
    "status",
    "has_annotations",
    "annotation_event_count",
    "identified_event_count",
    "detector_valid_event_count",
    "normalized_path",
    "duration_sec",
    "normalized_sample_rate",
    "normalized_channels",
    "audio_available",
]

recordings_base = (
    recordings_base[RECORDING_COLUMNS]
    .sort_values(
        ["dataset", "source_group", "source_recording"],
        na_position="last",
    )
    .reset_index(drop=True)
)

display(
    recordings_base.groupby("dataset").agg(
        audios_fisicos=("recording_uid", "size"),
        audios_anotados=("has_annotations", "sum"),
        horas=(
            "duration_sec",
            lambda x: x.sum() / 3600,
        ),
    )
)

# 9. Auditoría de geometría

In [ ]:
invalid_geometry = events_base.loc[
    ~events_base["use_detector"]
].copy()

print("Eventos totales:", len(events_base))
print(
    "Detector válidos:",
    int(events_base["use_detector"].sum()),
)
print(
    "Detector inválidos:",
    len(invalid_geometry),
)
print(
    "Identificados:",
    int(events_base["identified"].sum()),
)
print(
    "No identificados:",
    int((~events_base["identified"]).sum()),
)

display(
    events_base.groupby("dataset").agg(
        eventos=("event_id", "size"),
        identificados=("identified", "sum"),
        no_identificados=(
            "identified",
            lambda x: int((~x).sum()),
        ),
        detector_validos=("use_detector", "sum"),
        detector_invalidos=(
            "use_detector",
            lambda x: int((~x).sum()),
        ),
    )
)

if len(invalid_geometry):
    display(
        invalid_geometry[
            [
                "dataset",
                "recording",
                "event_id",
                "t_min",
                "t_max",
                "f_min_original",
                "f_max_original",
                "f_min",
                "f_max",
            ]
        ]
        .sort_values(
            ["dataset", "recording", "event_id"]
        )
        .reset_index(drop=True)
    )

# 10. Validaciones fuertes

Si cualquiera falla, no se deben publicar los manifests.

In [ ]:
checks = []

def check(name, condition, observed, expected):
    ok = bool(condition)

    checks.append({
        "check": name,
        "ok": ok,
        "observed": observed,
        "expected": expected,
    })

    if not ok:
        raise AssertionError(
            f"{name}: observed={observed}, expected={expected}"
        )


check(
    "eventos_totales",
    len(events_base) == 48465,
    len(events_base),
    48465,
)

check(
    "event_id_unicos",
    events_base["event_id"].is_unique,
    int(events_base["event_id"].nunique()),
    48465,
)

check(
    "grabaciones_anotadas_totales",
    events_base["recording_uid"].nunique() == 1011,
    int(events_base["recording_uid"].nunique()),
    1011,
)

expected_events = {
    "PteroSet": 15372,
    "Col-CR": 7338,
    "Peru": 16482,
    "WABAD-CO": 9273,
}

expected_annotated = {
    "PteroSet": 563,
    "Col-CR": 34,
    "Peru": 21,
    "WABAD-CO": 393,
}

expected_physical = {
    "PteroSet": 563,
    "Col-CR": 34,
    "Peru": 21,
    "WABAD-CO": 399,
}

for dataset, expected in expected_events.items():
    observed = int(
        events_base["dataset"].eq(dataset).sum()
    )
    check(
        f"{dataset}_eventos",
        observed == expected,
        observed,
        expected,
    )

for dataset, expected in expected_annotated.items():
    observed = int(
        events_base.loc[
            events_base["dataset"].eq(dataset),
            "recording_uid",
        ].nunique()
    )
    check(
        f"{dataset}_grabaciones_anotadas",
        observed == expected,
        observed,
        expected,
    )

for dataset, expected in expected_physical.items():
    observed = int(
        recordings_base["dataset"].eq(dataset).sum()
    )
    check(
        f"{dataset}_audios_fisicos",
        observed == expected,
        observed,
        expected,
    )

check(
    "audios_fisicos_totales",
    len(recordings_base) == 1017,
    len(recordings_base),
    1017,
)

check(
    "audios_anotados",
    int(recordings_base["has_annotations"].sum()) == 1011,
    int(recordings_base["has_annotations"].sum()),
    1011,
)

check(
    "audios_solo_fisicos",
    int((~recordings_base["has_annotations"]).sum()) == 6,
    int((~recordings_base["has_annotations"]).sum()),
    6,
)

check(
    "eventos_identificados",
    int(events_base["identified"].sum()) == 37725,
    int(events_base["identified"].sum()),
    37725,
)

check(
    "eventos_no_identificados",
    int((~events_base["identified"]).sum()) == 10740,
    int((~events_base["identified"]).sum()),
    10740,
)

check(
    "detector_validos",
    int(events_base["use_detector"].sum()) == 48430,
    int(events_base["use_detector"].sum()),
    48430,
)

check(
    "detector_invalidos",
    int((~events_base["use_detector"]).sum()) == 35,
    int((~events_base["use_detector"]).sum()),
    35,
)

annotated_uids = set(events_base["recording_uid"])
physical_uids = set(recordings_base["recording_uid"])

check(
    "ninguna_grabacion_anotada_sin_audio",
    annotated_uids <= physical_uids,
    len(annotated_uids - physical_uids),
    0,
)

check(
    "recording_uid_fisicos_unicos",
    recordings_base["recording_uid"].is_unique,
    int(recordings_base["recording_uid"].nunique()),
    1017,
)

check(
    "todos_audio_32khz_mono",
    (
        (
            recordings_base["normalized_sample_rate"]
            == 32000
        ).all()
        and (
            recordings_base["normalized_channels"]
            == 1
        ).all()
    ),
    "all valid",
    "all valid",
)

integrity_report = pd.DataFrame(checks)

display(integrity_report)

print("\nTodas las validaciones fuertes pasaron.")

# 11. Resumen base

In [ ]:
event_summary = (
    events_base.groupby("dataset")
    .agg(
        annotated_recordings=("recording_uid", "nunique"),
        events=("event_id", "size"),
        identified_events=("identified", "sum"),
        detector_valid_events=("use_detector", "sum"),
    )
)

physical_summary = (
    recordings_base.groupby("dataset")
    .agg(
        physical_recordings=("recording_uid", "size"),
        hours=(
            "duration_sec",
            lambda x: x.sum() / 3600,
        ),
    )
)

dataset_base_summary = (
    physical_summary
    .join(event_summary, how="outer")
    .reset_index()
)

dataset_base_summary["audio_only_recordings"] = (
    dataset_base_summary["physical_recordings"]
    - dataset_base_summary["annotated_recordings"]
)

dataset_base_summary["unidentified_events"] = (
    dataset_base_summary["events"]
    - dataset_base_summary["identified_events"]
)

dataset_base_summary["detector_invalid_events"] = (
    dataset_base_summary["events"]
    - dataset_base_summary["detector_valid_events"]
)

integer_columns = [
    "physical_recordings",
    "annotated_recordings",
    "audio_only_recordings",
    "events",
    "identified_events",
    "unidentified_events",
    "detector_valid_events",
    "detector_invalid_events",
]

for column in integer_columns:
    dataset_base_summary[column] = (
        dataset_base_summary[column]
        .astype(int)
    )

display(dataset_base_summary)

# 12. Guardado

Los archivos se escriben únicamente después de pasar las validaciones.

In [ ]:
events_base.to_csv(
    EVENTS_BASE_PATH,
    index=False,
)

recordings_base.to_csv(
    RECORDINGS_BASE_PATH,
    index=False,
)

dataset_base_summary.to_csv(
    STATISTICS_DIR / "dataset_base_summary.csv",
    index=False,
)

integrity_report.to_csv(
    STATISTICS_DIR / "integrity_base_report.csv",
    index=False,
)

print("Archivos generados:\n")

for path in [
    EVENTS_BASE_PATH,
    RECORDINGS_BASE_PATH,
    STATISTICS_DIR / "audio_source_inventory.csv",
    STATISTICS_DIR / "audio_annotation_reconciliation.csv",
    STATISTICS_DIR / "dataset_base_summary.csv",
    STATISTICS_DIR / "integrity_base_report.csv",
]:
    print(path)

# Resultado esperado

Al finalizar correctamente:

```text
Eventos anotados:                        48 465
Grabaciones anotadas:                     1 011
Audios físicos normalizados:              1 017
Audios WABAD-CO sin anotaciones:              6

Eventos identificados:                   37 725
Eventos no identificados:                10 740

Eventos válidos para detector:           48 430
Eventos con geometría inválida:              35
```
